# 04 - Analisis Exploratorio de Datos (EDA)

**Objetivo**: Entender la estructura de los datos, encontrar patrones y anomalias.

Analisis:
1. Distribucion general del dataset
2. Histogramas de variables clave
3. Matriz de correlacion
4. Patrones temporales
5. Comparacion entre cadenas

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import config

# Configuracion de graficos
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Librerias cargadas")

In [ ]:
# Cargar datos limpios de Parquet
processed = config.PROCESSED_DIR

try:
    tokens_df = pd.read_parquet(processed / "tokens_clean.parquet")
    print(f"Tokens: {len(tokens_df)}")
except FileNotFoundError:
    print("⚠️ No se encontro tokens_clean.parquet. Ejecuta notebook 03 primero.")
    tokens_df = pd.DataFrame()

try:
    ohlcv_df = pd.read_parquet(processed / "ohlcv_clean.parquet")
    print(f"OHLCV: {len(ohlcv_df)}")
except FileNotFoundError:
    ohlcv_df = pd.DataFrame()

try:
    snapshots_df = pd.read_parquet(processed / "snapshots_clean.parquet")
    print(f"Snapshots: {len(snapshots_df)}")
except FileNotFoundError:
    snapshots_df = pd.DataFrame()

## 1. Distribucion General

In [ ]:
if not tokens_df.empty:
    # Tokens por cadena
    fig = px.pie(
        tokens_df, 
        names='chain', 
        title='Distribucion de Tokens por Cadena',
        color_discrete_sequence=px.colors.qualitative.Set2
    )
    fig.show()
else:
    print("No hay datos de tokens")

In [ ]:
if not tokens_df.empty and 'dex' in tokens_df.columns:
    # Tokens por DEX
    dex_counts = tokens_df['dex'].value_counts().head(10)
    fig = px.bar(
        x=dex_counts.index, 
        y=dex_counts.values,
        title='Top 10 DEX por Numero de Tokens',
        labels={'x': 'DEX', 'y': 'Numero de Tokens'}
    )
    fig.show()

## 2. Distribucion de Precio, Volumen y Liquidez

In [ ]:
if not snapshots_df.empty:
    # Distribucion de liquidez (log scale)
    liq_data = snapshots_df['liquidity_usd'].dropna()
    liq_data = liq_data[liq_data > 0]
    
    fig = px.histogram(
        x=np.log10(liq_data), 
        nbins=50,
        title='Distribucion de Liquidez (log10 USD)',
        labels={'x': 'log10(Liquidez USD)', 'y': 'Frecuencia'}
    )
    fig.show()
    
    print(f"Liquidez mediana: ${liq_data.median():,.0f}")
    print(f"Liquidez media: ${liq_data.mean():,.0f}")
    print(f"Liquidez min: ${liq_data.min():,.2f}")
    print(f"Liquidez max: ${liq_data.max():,.0f}")

In [ ]:
if not snapshots_df.empty:
    # Distribucion de volumen 24h (log scale)
    vol_data = snapshots_df['volume_24h'].dropna()
    vol_data = vol_data[vol_data > 0]
    
    fig = px.histogram(
        x=np.log10(vol_data), 
        nbins=50,
        title='Distribucion de Volumen 24h (log10 USD)',
        labels={'x': 'log10(Volumen 24h USD)', 'y': 'Frecuencia'}
    )
    fig.show()

## 3. Analisis de Buyers vs Sellers

In [ ]:
if not snapshots_df.empty and 'buyers_24h' in snapshots_df.columns:
    # Scatter: buyers vs sellers
    buyer_seller = snapshots_df[['buyers_24h', 'sellers_24h']].dropna()
    buyer_seller = buyer_seller[(buyer_seller > 0).all(axis=1)]
    
    fig = px.scatter(
        buyer_seller,
        x='buyers_24h',
        y='sellers_24h',
        title='Compradores vs Vendedores (24h)',
        labels={'buyers_24h': 'Compradores', 'sellers_24h': 'Vendedores'},
        opacity=0.5,
    )
    # Linea diagonal (ratio 1:1)
    max_val = max(buyer_seller['buyers_24h'].max(), buyer_seller['sellers_24h'].max())
    fig.add_trace(go.Scatter(
        x=[0, max_val], y=[0, max_val],
        mode='lines', name='Ratio 1:1',
        line=dict(dash='dash', color='red')
    ))
    fig.show()
    
    # Ratio buyer/seller
    ratios = buyer_seller['buyers_24h'] / buyer_seller['sellers_24h']
    print(f"Ratio buyer/seller mediano: {ratios.median():.2f}")
    print(f"Tokens con mas compradores que vendedores: {(ratios > 1).mean()*100:.1f}%")

## 4. Patrones Temporales

In [ ]:
if not tokens_df.empty and 'created_at' in tokens_df.columns:
    tokens_with_date = tokens_df.dropna(subset=['created_at']).copy()
    tokens_with_date['created_at'] = pd.to_datetime(tokens_with_date['created_at'])
    
    if not tokens_with_date.empty:
        # Tokens creados por dia de la semana
        tokens_with_date['day_of_week'] = tokens_with_date['created_at'].dt.day_name()
        day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
        day_counts = tokens_with_date['day_of_week'].value_counts().reindex(day_order)
        
        fig = px.bar(
            x=day_counts.index, y=day_counts.values,
            title='Tokens Creados por Dia de la Semana',
            labels={'x': 'Dia', 'y': 'Numero de Tokens'}
        )
        fig.show()
        
        # Tokens creados por hora UTC
        tokens_with_date['hour_utc'] = tokens_with_date['created_at'].dt.hour
        hour_counts = tokens_with_date['hour_utc'].value_counts().sort_index()
        
        fig = px.bar(
            x=hour_counts.index, y=hour_counts.values,
            title='Tokens Creados por Hora UTC',
            labels={'x': 'Hora UTC', 'y': 'Numero de Tokens'}
        )
        fig.show()

## 5. Correlaciones entre Variables Numericas

In [ ]:
if not snapshots_df.empty:
    # Seleccionar columnas numericas
    numeric_cols = snapshots_df.select_dtypes(include=[np.number]).columns.tolist()
    # Excluir columnas de ID
    numeric_cols = [c for c in numeric_cols if c != 'id']
    
    if len(numeric_cols) >= 3:
        corr_matrix = snapshots_df[numeric_cols].corr()
        
        fig = px.imshow(
            corr_matrix,
            text_auto='.2f',
            title='Matriz de Correlacion - Pool Snapshots',
            color_continuous_scale='RdBu_r',
            zmin=-1, zmax=1,
        )
        fig.update_layout(width=800, height=700)
        fig.show()
        
        # Top correlaciones
        print("\nTop correlaciones (excluyendo diagonales):")
        corr_pairs = []
        for i in range(len(corr_matrix.columns)):
            for j in range(i+1, len(corr_matrix.columns)):
                corr_pairs.append({
                    'var1': corr_matrix.columns[i],
                    'var2': corr_matrix.columns[j],
                    'corr': abs(corr_matrix.iloc[i, j]),
                })
        corr_pairs = sorted(corr_pairs, key=lambda x: x['corr'], reverse=True)
        for p in corr_pairs[:10]:
            print(f"  {p['var1']} <-> {p['var2']}: {p['corr']:.3f}")

## 6. Resumen EDA

### Hallazgos clave:
- (Completar despues de ejecutar las celdas anteriores)

### Siguiente paso:
Ejecutar `05_feature_engineering.ipynb` para construir la matriz de features.